# Graded Mini Project: Part A

# City Library Management System

**Submitted by: Venkat Phani Kiran**

**Date: 12th Sep 2026**

---

## Project Overview

This prototype manages a community library's book inventory, member records, borrowing and return transactions, searches and reports.

The program reads Books_data.csv from Google Drive, stores books and members in dictionaries and the borrow log in a list, and performs every library operation through a small set of reusable functions. The sample runs show successful actions, rejected requests, record updates and final reports.

---

## Workflow

| Step | Section | What happens |
|---|---|---|
| 1 | Data Preparation | Load Books_data.csv from Google Drive, inspect and clean it, convert it into the book dictionary |
| 2 | Book Records and Validation | Add new books after validating the Book ID, title, author and genre |
| 3 | Member Records | Register members after validating the Member ID, name, age and contact |
| 4 | Borrow and Return System | Issue a book only when the member exists and the book is available; return a book only against a matching open loan |
| 5 | Reports and Queries | Available books by genre, current borrowers, title or author search, most popular issued genre |
| 6 | Connected Sample Run and Final Summary | One complete add, borrow, return sequence, then the library summary and reconciliation of totals |
| 7 | Technical Notes | Design decisions and assumptions |
| 8 | Conclusion and Limitations | Findings of the sample run and what the prototype does not cover |
| Appendix A | Optional Interactive Run | Check availability and add a book through keyboard input |

---

## Function Map

| Function | Purpose |
|---|---|
| valid_id(value) | Accepts only positive whole numbers, used for IDs and ages |
| valid_text(value) | Accepts only non-empty text |
| add_book(book_id, title, author, genre) | Adds a validated book with Available status |
| register_member(member_id, name, age, contact) | Adds a validated member |
| active_loans(member_id=None) | Lists open loans, for everyone or for one member |
| borrow_book(member_id, book_id) | Issues an available book to a registered member and writes a borrow-log entry |
| return_book(member_id, book_id) | Returns a book against its open loan and makes it available again |
| show_table(records) | Prints a list of records as a table, or a clear message when there are none |
| available_by_genre(genre) | Lists available books in a genre |
| members_with_books() | Lists members who currently hold books |
| search_books(query) | Finds books by full or partial title or author |
| most_popular_genre() | Finds the genre or genres with the most issued books |
| library_summary() | Totals for books, members and loans, with the most popular genre |
| check_book_availability(query) | Appendix: shows the availability of books matching a title or author |
| add_book_from_input() | Appendix: collects a new book from keyboard input and adds it |

## 1. Data Preparation

Load the book inventory from Google Drive, look at its size, types and labels, clean the text fields, and convert the table into the book dictionary that the rest of the program uses.

### 1.1 Connect Google Drive

In [6]:
# Import pandas to read and inspect the CSV file.
import pandas as pd
# Import the Google Drive connection tool available in Colab.
from google.colab import drive

# Connect this notebook to Google Drive.
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1.2 Read the CSV file and preview the records

In [7]:
# Store the Drive folder of the course.
COURSE_FOLDER = '/content/drive/MyDrive/IITM AI & Agentic Course Aug 2026/Colab Notebooks/'
# Add the assignment sub-folder and the file name.
BOOKS_CSV = COURSE_FOLDER + 'Assignment_1_W4/Part A/Books_data.csv'
# Read the CSV file into a table.
source_df = pd.read_csv(BOOKS_CSV)
# Show the first five rows.
source_df.head()

,bid,title,author,category,status
0,1,Steve Jobs,Walter Issacson,Biography,issued
1,2,Discovery of India,Jawaharlal Nehru,History,issued
2,3,My Experiments with Truth,Mahatma Gandhi,Autobiography,issued
3,4,Object Oriented Programming with C++,E Balagurusamy,Education,issued
4,5,Thinking with type,Ellen Lupton,Arts,issued


### 1.3 Check the number of rows and columns

In [8]:
# Print the number of rows and columns of the table.
print('Rows and columns:', source_df.shape)

Rows and columns: (61, 5)


### 1.4 Check the column names and data types

In [9]:
# Show the column names, their data types and the count of filled values.
source_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   bid       61 non-null     int64 
 1   title     61 non-null     object
 2   author    61 non-null     object
 3   category  61 non-null     object
 4   status    61 non-null     object
dtypes: int64(1), object(4)
memory usage: 2.5+ KB


### 1.5 Check for missing values

In [10]:
# Count the missing values in each column.
source_df.isna().sum()

,0
bid,0
title,0
author,0
category,0
status,0


### 1.6 Check for duplicate book IDs

In [11]:
# Count book IDs that appear more than once.
print('Duplicate book IDs:', source_df['bid'].duplicated().sum())

Duplicate book IDs: 0


### 1.7 Check the availability labels

In [12]:
# Count how many books carry each status label.
source_df['status'].value_counts()

,count
status,
available,39
issued,22


### 1.8 Check the genre labels

In [13]:
# Count how many books belong to each genre label.
source_df['category'].value_counts()

,count
category,
Fiction,10
Biography,6
Autobiography,6
History,6
Story,6
Economics,6
Philosophy,5
Education,3
Personal Development,2


### 1.9 Rename the columns and clean the text

In [14]:
# Make a copy so the original table stays untouched.
book_df = source_df.copy()
# Match the CSV column names to the names used in this program.
new_names = {'bid': 'book_id', 'category': 'genre', 'status': 'availability'}
# Rename the columns.
book_df = book_df.rename(columns=new_names)
# List the text columns that need cleaning.
text_columns = ['title', 'author', 'genre', 'availability']
# Go through the text columns one at a time.
for column in text_columns:
    # Remove spaces at the start and end of every value in this column.
    book_df[column] = book_df[column].str.strip()
# Write the availability labels in lowercase so they compare correctly.
book_df['availability'] = book_df['availability'].str.lower()
# Show the cleaned table.
book_df.head()

,book_id,title,author,genre,availability
0,1,Steve Jobs,Walter Issacson,Biography,issued
1,2,Discovery of India,Jawaharlal Nehru,History,issued
2,3,My Experiments with Truth,Mahatma Gandhi,Autobiography,issued
3,4,Object Oriented Programming with C++,E Balagurusamy,Education,issued
4,5,Thinking with type,Ellen Lupton,Arts,issued


### 1.10 Run the source checks

In [15]:
# Check that no value is missing.
print('No missing values:', book_df.isna().sum().sum() == 0)
# Check that every book ID is unique.
print('Book IDs are unique:', book_df['book_id'].is_unique)
# Check that every book ID is a positive number.
print('Book IDs are positive:', (book_df['book_id'] > 0).all())
# Check that every availability label is either available or issued.
print('Only allowed labels:', book_df['availability'].isin(['available', 'issued']).all())

No missing values: True
Book IDs are unique: True
Book IDs are positive: True
Only allowed labels: True


### 1.11 Build the library data structures

In [16]:
# Start with an empty dictionary of books.
books = {}
# Go through the cleaned table one row at a time.
for index, row in book_df.iterrows():
    # Create the record for this book.
    book = {}
    # Copy the title.
    book['title'] = row['title']
    # Copy the author.
    book['author'] = row['author']
    # Copy the genre.
    book['genre'] = row['genre']
    # Copy the availability label.
    book['availability'] = row['availability']
    # Store the record under the book ID, kept as a plain whole number.
    books[int(row['book_id'])] = book
# Start with no members; the CSV has book data only.
members = {}
# Start with an empty borrow log; it records every new loan.
borrow_log = []
# Count the books that were already issued in the CSV; their borrowers are not known.
source_issued = int((book_df['availability'] == 'issued').sum())
# Print the starting counts.
print('Books:', len(books), '| Members:', len(members), '| Loans in log:', len(borrow_log))
# Print how many source books are issued without borrower details.
print('Books issued in the source CSV:', source_issued)

Books: 61 | Members: 0 | Loans in log: 0
Books issued in the source CSV: 22


**Preparation findings.** The source has 61 books: 39 available and 22 issued. There are no missing values and no duplicate IDs. The columns bid, category and status are renamed to book_id, genre and availability, outer spaces are removed and the availability labels are written in lowercase. Title, author and genre spellings are kept as supplied. The 22 books already issued in the CSV have no borrower or transaction details, so they stay issued and no loan records are invented for them.

## 2. Book Records and Validation

Define the two validation helpers, then add a new book and show that duplicate IDs, blank details and wrong ID types are refused without changing any record.

### 2.1 Define the rule for IDs and ages

In [17]:
# Define the rule for IDs and ages: a positive whole number.
def valid_id(value):
    # Accept only the int type; this also rejects True and False.
    if type(value) != int:
        # Anything that is not an int fails.
        return False
    # Accept only values above zero.
    return value > 0
# Test with a valid ID.
print('1001 is valid:', valid_id(1001))

1001 is valid: True


### 2.2 Reject invalid IDs

In [18]:
# Test with text digits; text must fail even when it looks like a number.
print("'1001' is valid:", valid_id('1001'))
# Test with zero.
print('0 is valid:', valid_id(0))
# Test with a Boolean value.
print('True is valid:', valid_id(True))

'1001' is valid: False
0 is valid: False
True is valid: False


### 2.3 Define the rule for required text

In [19]:
# Define the rule for names and book details: non-empty text.
def valid_text(value):
    # Accept only the str type.
    if type(value) != str:
        # Numbers, None and other types fail.
        return False
    # Accept only text that is not blank after removing outer spaces.
    return value.strip() != ''
# Test a normal title.
print("'Python Practice' is valid:", valid_text('Python Practice'))
# Test a blank title.
print("'   ' is valid:", valid_text('   '))

'Python Practice' is valid: True
'   ' is valid: False


### 2.4 Add a new book

In [20]:
# Define the operation that adds one book to the inventory.
def add_book(book_id, title, author, genre):
    # Check the ID first.
    if not valid_id(book_id):
        # Refuse a wrong ID without changing anything.
        return 'Rejected: Book ID must be a positive whole number.'
    # Check that title, author and genre are all filled.
    if not valid_text(title) or not valid_text(author) or not valid_text(genre):
        # Refuse blank details.
        return 'Rejected: title, author and genre must all be filled.'
    # Check that the ID is not already used.
    if book_id in books:
        # Refuse to overwrite an existing book.
        return 'Rejected: Book ID already exists.'
    # Create the record for the new book.
    book = {}
    # Store the trimmed title.
    book['title'] = title.strip()
    # Store the trimmed author.
    book['author'] = author.strip()
    # Store the trimmed genre.
    book['genre'] = genre.strip()
    # A new book always starts as available.
    book['availability'] = 'available'
    # Save the record in the inventory.
    books[book_id] = book
    # Confirm the addition.
    return 'Added book ' + str(book_id) + ': ' + book['title'] + '.'
# Add a fictional practice book.
print(add_book(1001, 'Python Practice', 'Practice Author', 'Education'))

Added book 1001: Python Practice.


### 2.5 Inspect the added record

In [21]:
# Show the record that was just added.
print(books[1001])

{'title': 'Python Practice', 'author': 'Practice Author', 'genre': 'Education', 'availability': 'available'}


### 2.6 Reject a duplicate Book ID

In [22]:
# Attempt to add a different book using an ID that already exists.
print(add_book(1001, 'Replacement Title', 'Another Author', 'Fiction'))

Rejected: Book ID already exists.


### 2.7 Reject blank book details

In [23]:
# Attempt to add a book with a blank title.
print(add_book(1002, '   ', 'Practice Author', 'Education'))

Rejected: title, author and genre must all be filled.


### 2.8 Reject a Boolean or text Book ID

In [24]:
# Attempt to add a book with True as the ID.
print(add_book(True, 'Some Title', 'Some Author', 'Fiction'))
# Attempt to add a book with the ID written as text.
print(add_book('1003', 'Some Title', 'Some Author', 'Fiction'))

Rejected: Book ID must be a positive whole number.
Rejected: Book ID must be a positive whole number.


### 2.9 Confirm that rejected requests changed nothing

In [25]:
# Confirm the original record kept its title.
print('Book 1001 unchanged:', books[1001]['title'] == 'Python Practice')
# Confirm the rejected IDs were not created.
print('Rejected books not saved:', 1002 not in books and '1003' not in books)
# Print the new inventory size.
print('Total books:', len(books))

Book 1001 unchanged: True
Rejected books not saved: True
Total books: 62


**Book validation findings.** The new book is added as available. A duplicate ID, a blank title, a Boolean ID and a text ID are all refused before any record changes, and the original record keeps its title. The inventory grows from 61 to 62 books.

## 3. Member Records

Register two fictional members and show that a repeated ID, an invalid age and a missing contact are refused without changing the member records.

### 3.1 Register a member

In [26]:
# Define the operation that registers one member.
def register_member(member_id, name, age, contact):
    # Check the member ID and the age; both must be positive whole numbers.
    if not valid_id(member_id) or not valid_id(age):
        # Refuse wrong IDs or ages.
        return 'Rejected: Member ID and age must be positive whole numbers.'
    # Check the name and the contact.
    if not valid_text(name) or not valid_text(contact):
        # Refuse a blank name or contact.
        return 'Rejected: name and contact must both be filled.'
    # Check that the ID is free.
    if member_id in members:
        # Refuse to overwrite an existing member.
        return 'Rejected: Member ID already exists.'
    # Create the member record.
    member = {}
    # Store the trimmed name.
    member['name'] = name.strip()
    # Store the age.
    member['age'] = age
    # Store the trimmed contact.
    member['contact'] = contact.strip()
    # Save the record.
    members[member_id] = member
    # Confirm the registration.
    return 'Registered member ' + str(member_id) + ': ' + member['name'] + '.'
# Register the first fictional member.
print(register_member(101, 'Asha Rao', 21, 'asha@example.com'))

Registered member 101: Asha Rao.


### 3.2 Register a second member

In [27]:
# Register a second member for the wrong-member and second-borrower tests later.
print(register_member(102, 'Ravi Shah', 25, 'ravi@example.com'))

Registered member 102: Ravi Shah.


### 3.3 Inspect a member record

In [28]:
# Show all stored fields of member 101.
print(members[101])

{'name': 'Asha Rao', 'age': 21, 'contact': 'asha@example.com'}


### 3.4 Reject a duplicate Member ID

In [29]:
# Attempt to reuse an existing member ID.
print(register_member(101, 'Another Name', 30, 'other@example.com'))

Rejected: Member ID already exists.


### 3.5 Reject an invalid age

In [30]:
# Attempt to register with age zero.
print(register_member(103, 'Test Member', 0, 'test@example.com'))
# Attempt to register with a Boolean age.
print(register_member(103, 'Test Member', True, 'test@example.com'))

Rejected: Member ID and age must be positive whole numbers.
Rejected: Member ID and age must be positive whole numbers.


### 3.6 Reject a missing contact

In [31]:
# Attempt to register with no contact information.
print(register_member(104, 'Sample Member', 22, None))

Rejected: name and contact must both be filled.


### 3.7 Confirm the member records

In [32]:
# Confirm only the two valid members exist.
print('Registered members:', len(members))
# Confirm the rejected IDs were not created.
print('Rejected members not saved:', 103 not in members and 104 not in members)
# Confirm the first member record is unchanged.
print('Member 101 unchanged:', members[101]['name'] == 'Asha Rao')

Registered members: 2
Rejected members not saved: True
Member 101 unchanged: True


**Member validation findings.** Two fictional members are registered. A repeated ID, an age of zero, a Boolean age and a missing contact are all refused, and the existing records stay unchanged.

## 4. Borrow and Return System

Issue books, refuse the requests that must fail, return a book against its own loan, keep the completed loan in the log and issue the returned book again.

### 4.1 Define the open-loan query

In [33]:
# Define a query that lists the loans not yet returned.
def active_loans(member_id=None):
    # Start with an empty list of results.
    result = []
    # Look at every loan in the log.
    for loan in borrow_log:
        # Keep only loans that are still open.
        if not loan['returned']:
            # Keep the loan when no member is given, or when the member matches.
            if member_id is None or loan['member_id'] == member_id:
                # Add the loan to the results.
                result.append(loan)
    # Give back the open loans.
    return result
# Show the open loans before any borrowing.
print(active_loans())

[]


### 4.2 Borrow an available book

In [34]:
# Define the operation that issues a book to a member.
def borrow_book(member_id, book_id):
    # Both IDs must be positive whole numbers.
    if not valid_id(member_id) or not valid_id(book_id):
        # Refuse wrong ID types.
        return 'Rejected: IDs must be positive whole numbers.'
    # The member must exist.
    if member_id not in members:
        # Refuse unknown members.
        return 'Rejected: member not found.'
    # The book must exist.
    if book_id not in books:
        # Refuse unknown books.
        return 'Rejected: book not found.'
    # The book must be available.
    if books[book_id]['availability'] != 'available':
        # Refuse books that are already issued.
        return 'Rejected: book is already issued.'
    # Create the loan record.
    loan = {}
    # Number the loan in order of creation.
    loan['loan_id'] = len(borrow_log) + 1
    # Record the borrowing member.
    loan['member_id'] = member_id
    # Record the borrowed book.
    loan['book_id'] = book_id
    # Mark the loan as open.
    loan['returned'] = False
    # Add the loan to the borrow log.
    borrow_log.append(loan)
    # Mark the book as issued.
    books[book_id]['availability'] = 'issued'
    # Confirm the issue.
    return 'Issued book ' + str(book_id) + ' to member ' + str(member_id) + '.'
# Issue the new practice book to member 101.
print(borrow_book(101, 1001))

Issued book 1001 to member 101.


### 4.3 Check the updated book record

In [35]:
# Show the availability changed by the borrowing function.
print('Book 1001 is now:', books[1001]['availability'])

Book 1001 is now: issued


### 4.4 Check the new borrow-log entry

In [36]:
# Show the latest loan recorded in the log.
print(borrow_log[-1])

{'loan_id': 1, 'member_id': 101, 'book_id': 1001, 'returned': False}


### 4.5 Allow several books per member

In [37]:
# Issue a second, different available book to the same member.
print(borrow_book(101, 14))

Issued book 14 to member 101.


### 4.6 Confirm both open loans

In [38]:
# Show both open loans of member 101.
print(active_loans(101))

[{'loan_id': 1, 'member_id': 101, 'book_id': 1001, 'returned': False}, {'loan_id': 2, 'member_id': 101, 'book_id': 14, 'returned': False}]


### 4.7 Reject borrowing an already issued book

In [39]:
# Attempt to issue the same copy to another member.
print(borrow_book(102, 1001))

Rejected: book is already issued.


### 4.8 Reject an unknown member

In [40]:
# Attempt to issue an available book to a member who is not registered.
print(borrow_book(999, 15))

Rejected: member not found.


### 4.9 Reject an unknown book

In [41]:
# Attempt to issue a book ID that is not in the inventory.
print(borrow_book(101, 9999))

Rejected: book not found.


### 4.10 Respect the issued status from the source CSV

In [42]:
# Attempt to issue a book that the CSV already marks as issued.
print(borrow_book(101, 1))

Rejected: book is already issued.


### 4.11 Confirm that failed issues changed nothing

In [43]:
# Confirm only the two successful loans are in the log.
print('Loans in log:', len(borrow_log))
# Confirm the untouched book 15 is still available.
print('Book 15 is still:', books[15]['availability'])

Loans in log: 2
Book 15 is still: available


### 4.12 Return a book

In [44]:
# Define the operation that takes a book back from a member.
def return_book(member_id, book_id):
    # Both IDs must be positive whole numbers.
    if not valid_id(member_id) or not valid_id(book_id):
        # Refuse wrong ID types.
        return 'Rejected: IDs must be positive whole numbers.'
    # The member and the book must exist.
    if member_id not in members or book_id not in books:
        # Refuse unknown members or books.
        return 'Rejected: member or book not found.'
    # Start with no matching loan found.
    matching_loan = None
    # Look through this member's open loans.
    for loan in active_loans(member_id):
        # Check whether this loan is for the wanted book.
        if loan['book_id'] == book_id:
            # Remember the loan of this book.
            matching_loan = loan
    # Refuse when this member has no open loan of this book.
    if matching_loan is None:
        # This covers the wrong member, a repeated return and books issued outside the log.
        return 'Rejected: no open loan of this book by this member.'
    # Close the loan but keep it in the log as history.
    matching_loan['returned'] = True
    # Make the book available again.
    books[book_id]['availability'] = 'available'
    # Confirm the return.
    return 'Returned book ' + str(book_id) + ' from member ' + str(member_id) + '.'
# Attempt a return by the wrong member.
print(return_book(102, 1001))

Rejected: no open loan of this book by this member.


### 4.13 Check the loans after the rejected return

In [45]:
# Confirm both loans still belong to member 101.
print(active_loans(101))

[{'loan_id': 1, 'member_id': 101, 'book_id': 1001, 'returned': False}, {'loan_id': 2, 'member_id': 101, 'book_id': 14, 'returned': False}]


### 4.14 Complete the correct return

In [46]:
# Return the book through the member who borrowed it.
print(return_book(101, 1001))

Returned book 1001 from member 101.


### 4.15 Check the returned book

In [47]:
# Show the availability after the return.
print('Book 1001 is now:', books[1001]['availability'])

Book 1001 is now: available


### 4.16 Check that the loan history is kept

In [48]:
# Show the first loan after its return.
print(borrow_log[0])

{'loan_id': 1, 'member_id': 101, 'book_id': 1001, 'returned': True}


### 4.17 Reject a repeated return

In [49]:
# Attempt to return the same book again.
print(return_book(101, 1001))

Rejected: no open loan of this book by this member.


### 4.18 Reject the return of a book issued outside the log

In [50]:
# Attempt to return a book that the CSV marks as issued but whose borrower is unknown.
print(return_book(101, 1))

Rejected: no open loan of this book by this member.


### 4.19 Issue the returned book again

In [51]:
# Issue the now available book to the other member.
print(borrow_book(102, 1001))

Issued book 1001 to member 102.


### 4.20 Confirm the loan history and open loans

In [52]:
# Count the returned loans in the log.
returned_count = 0
# Look at every loan.
for loan in borrow_log:
    # Check whether this loan is closed.
    if loan['returned']:
        # Add one to the count of returned loans.
        returned_count = returned_count + 1
# Print the loans recorded in the whole borrow, return and reissue cycle.
print('Loans in log:', len(borrow_log))
# Print the split between open and returned loans.
print('Open loans:', len(active_loans()), '| Returned loans:', returned_count)

Loans in log: 3
Open loans: 2 | Returned loans: 1


**Transaction findings.** A member can hold several books. A second issue of the same copy, an unknown member, an unknown book and a book already issued in the CSV are all refused. A return needs the same member and book in an open loan, so the wrong member, a repeated return and the source-issued books are refused. The completed loan stays in the log and the returned book can be issued again: 3 loans recorded, 2 open and 1 returned.

## 5. Reports and Queries

Produce the four required reports: available books in a genre, members who hold books, a title or author search, and the most popular issued genre. Searches ignore letter case and outer spaces.

### 5.1 Show the available books in a genre

In [53]:
# Define a helper that prints a list of records as a table.
def show_table(records):
    # Handle the case of no records.
    if len(records) == 0:
        # Print a clear message instead of an empty table.
        print('No matching records.')
    # Otherwise build a table from the records.
    else:
        # Turn the list of dictionaries into a table and print it without row numbers.
        print(pd.DataFrame(records).to_string(index=False))
# Define the query for available books in one genre.
def available_by_genre(genre):
    # Start with an empty result list.
    result = []
    # Check for a blank or missing genre.
    if not valid_text(genre):
        # Refuse it by returning no matches.
        return result
    # Prepare the genre for comparison: no outer spaces, lowercase.
    wanted = genre.strip().lower()
    # Look at every book.
    for book_id, book in books.items():
        # Keep books that are available and belong to the wanted genre.
        if book['availability'] == 'available' and book['genre'].lower() == wanted:
            # Start the record with the book ID.
            record = {'book_id': book_id}
            # Copy the book fields into the record.
            record.update(book)
            # Add the record to the results.
            result.append(record)
    # Give back the matching books.
    return result
# Query Education with extra spaces and capital letters.
show_table(available_by_genre('  EDUCATION  '))

 book_id                                 title      author     genre availability
      15 Automate The Boring Stuff With Python Al Sweigart Education    available


### 5.2 Handle a genre with no matches

In [54]:
# Query a genre that is not in the inventory.
show_table(available_by_genre('Unknown Genre'))

No matching records.


### 5.3 List the members who currently hold books

In [55]:
# Define the report of members with open loans.
def members_with_books():
    # Start with an empty result list.
    result = []
    # Look at every registered member.
    for member_id, member in members.items():
        # Collect the IDs of the books this member currently holds.
        held_books = []
        # Look at this member's open loans.
        for loan in active_loans(member_id):
            # Add the book ID of each open loan.
            held_books.append(loan['book_id'])
        # Include only members holding at least one book.
        if len(held_books) > 0:
            # Build the report record.
            record = {}
            # Store the member ID.
            record['member_id'] = member_id
            # Store the member name.
            record['name'] = member['name']
            # Store the list of held books.
            record['book_ids'] = held_books
            # Add the record to the results.
            result.append(record)
    # Give back the report.
    return result
# Show the members who currently hold books.
show_table(members_with_books())

 member_id      name book_ids
       101  Asha Rao     [14]
       102 Ravi Shah   [1001]


### 5.4 Search by title

In [56]:
# Define a search that matches part of a title or an author name.
def search_books(query):
    # Start with an empty result list.
    result = []
    # Check for blank or missing search text.
    if not valid_text(query):
        # Refuse it by returning no matches.
        return result
    # Prepare the search text: no outer spaces, lowercase.
    wanted = query.strip().lower()
    # Look at every book.
    for book_id, book in books.items():
        # Check the title.
        in_title = wanted in book['title'].lower()
        # Check the author.
        in_author = wanted in book['author'].lower()
        # Keep the book when either field matches.
        if in_title or in_author:
            # Start the record with the book ID.
            record = {'book_id': book_id}
            # Copy the book fields into the record.
            record.update(book)
            # Add the record to the results.
            result.append(record)
    # Give back the matching books.
    return result
# Search for Python titles using extra spaces and capital letters.
show_table(search_books('  PYTHON  '))

 book_id                                 title          author     genre availability
      14                        Python For All     John Shovic Education       issued
      15 Automate The Boring Stuff With Python     Al Sweigart Education    available
    1001                       Python Practice Practice Author Education       issued


### 5.5 Search by author

In [57]:
# Search for an author without matching the spelling of capital letters.
show_table(search_books('al sweigart'))

 book_id                                 title      author     genre availability
      15 Automate The Boring Stuff With Python Al Sweigart Education    available


### 5.6 Handle an empty search

In [58]:
# Confirm that blank search text does not match the whole inventory.
show_table(search_books('   '))

No matching records.


### 5.7 Find the most popular genre among issued books

In [59]:
# Define the popularity report: the genre or genres with the most issued books.
def most_popular_genre():
    # Start with an empty count for each genre.
    genre_counts = {}
    # Look at every book.
    for book in books.values():
        # Count only issued books.
        if book['availability'] == 'issued':
            # Read the genre as written in the inventory.
            genre = book['genre']
            # Add one to this genre's count, starting from zero the first time it appears.
            genre_counts[genre] = genre_counts.get(genre, 0) + 1
    # Start with an empty result list.
    result = []
    # Handle a library with no issued books.
    if len(genre_counts) == 0:
        # There is no popular genre to report.
        return result
    # Find the highest count.
    highest = max(genre_counts.values())
    # Look at the genres in alphabetical order.
    for genre in sorted(genre_counts):
        # Keep every genre that reaches the highest count, so ties are all reported.
        if genre_counts[genre] == highest:
            # Add the genre and its count.
            result.append({'genre': genre, 'issued_books': highest})
    # Give back the winning genre or genres.
    return result
# Show the most popular genre for the current library state.
show_table(most_popular_genre())

    genre  issued_books
Economics             3
Education             3
  Fiction             3
  History             3


### 5.8 Confirm the report results

In [60]:
# Collect the IDs of the available Education books.
education_ids = []
# Look at the report records.
for book in available_by_genre('Education'):
    # Keep the ID.
    education_ids.append(book['book_id'])
# Print the IDs; only book 15 should be available at this point.
print('Available Education book IDs:', education_ids)
# Print the number of members holding books.
print('Members holding books:', len(members_with_books()))
# Print the number of books found for Python.
print('Books found for Python:', len(search_books('Python')))

Available Education book IDs: [15]
Members holding books: 2
Books found for Python: 3


**Report findings.** Book 15 is the only available Education book at this point, because books 14 and 1001 are on loan. Member 101 holds book 14 and member 102 holds book 1001. The Python search finds books 14, 15 and 1001 whatever their availability. Economics, Education, Fiction and History tie with three issued books each, and all four are reported. Popularity counts the books issued right now, including the 22 issued in the source CSV.

## 6. Connected Sample Run and Final Summary

Check that the records agree with each other, run one complete sequence (add a book, borrow it, refuse a second issue, return it), then print the library summary and reconcile the totals.

### 6.1 Check that the records agree with each other

In [61]:
# Collect the book IDs of all open loans.
open_book_ids = []
# Look at every open loan.
for loan in active_loans():
    # Keep the book ID.
    open_book_ids.append(loan['book_id'])
# Each book can be in at most one open loan.
print('One open loan per book:', len(open_book_ids) == len(set(open_book_ids)))
# Assume every open loan points to an issued book until a mismatch is found.
all_marked_issued = True
# Look at every book on loan.
for book_id in open_book_ids:
    # Check the book on loan.
    if books[book_id]['availability'] != 'issued':
        # Flag a book that is on loan but not marked issued.
        all_marked_issued = False
# Print the result of the status check.
print('Open loans point to issued books:', all_marked_issued)
# Assume every loan refers to known records until a mismatch is found.
all_known = True
# Look at every loan in the log.
for loan in borrow_log:
    # Check the member and the book of this loan.
    if loan['member_id'] not in members or loan['book_id'] not in books:
        # Flag a loan whose member or book is missing.
        all_known = False
# Print the result of the reference check.
print('All loans refer to known members and books:', all_known)

One open loan per book: True
Open loans point to issued books: True
All loans refer to known members and books: True


### 6.2 Add a demonstration book

In [62]:
# Add a second fictional book for the connected run.
print(add_book(1002, 'Library Demo', 'Practice Author', 'Fiction'))

Added book 1002: Library Demo.


### 6.3 Show the new book as available

In [63]:
# Find the demonstration book before borrowing it.
show_table(search_books('Library Demo'))

 book_id        title          author   genre availability
    1002 Library Demo Practice Author Fiction    available


### 6.4 Borrow the demonstration book

In [64]:
# Issue the demonstration book to member 101.
print(borrow_book(101, 1002))

Issued book 1002 to member 101.


### 6.5 Show the updated record

In [65]:
# Show the book after issuing.
print(books[1002])

{'title': 'Library Demo', 'author': 'Practice Author', 'genre': 'Fiction', 'availability': 'issued'}


### 6.6 Show the new borrowing transaction

In [66]:
# Show the loan recorded for book 1002.
print(borrow_log[-1])

{'loan_id': 4, 'member_id': 101, 'book_id': 1002, 'returned': False}


### 6.7 Refuse a second issue of the same book

In [67]:
# Attempt to borrow the demonstration book while it is on loan.
print(borrow_book(102, 1002))

Rejected: book is already issued.


### 6.8 Return the demonstration book

In [68]:
# Return the demonstration book through its borrower.
print(return_book(101, 1002))

Returned book 1002 from member 101.


### 6.9 Show the restored availability

In [69]:
# Show the availability after the return.
print('Book 1002 is now:', books[1002]['availability'])

Book 1002 is now: available


### 6.10 Show the completed loan kept in the log

In [70]:
# Show the closed loan of book 1002.
print(borrow_log[-1])

{'loan_id': 4, 'member_id': 101, 'book_id': 1002, 'returned': True}


### 6.11 Show the current borrowers

In [71]:
# Show who holds books after the demonstration return.
show_table(members_with_books())

 member_id      name book_ids
       101  Asha Rao     [14]
       102 Ravi Shah   [1001]


### 6.12 Confirm the totals of the sample run

In [72]:
# Count the returned loans.
returned_count = 0
# Look at every loan.
for loan in borrow_log:
    # Check whether this loan is closed.
    if loan['returned']:
        # Add one to the count of returned loans.
        returned_count = returned_count + 1
# Print the record counts: 61 source books plus 2 added, 2 members, 4 loans.
print('Books:', len(books), '| Members:', len(members), '| Loans in log:', len(borrow_log))
# Print the loan split: 2 open and 2 returned.
print('Open loans:', len(active_loans()), '| Returned loans:', returned_count)

Books: 63 | Members: 2 | Loans in log: 4
Open loans: 2 | Returned loans: 2


### 6.13 Print the library summary

In [73]:
# Define the summary of the whole library.
def library_summary():
    # Start the count of available books at zero.
    available = 0
    # Start the count of issued books at zero.
    issued = 0
    # Look at every book.
    for book in books.values():
        # Check the availability label.
        if book['availability'] == 'available':
            # Count an available book.
            available = available + 1
        # Any other label means the book is issued.
        else:
            # Count an issued book.
            issued = issued + 1
    # Count the open loans.
    open_loans = len(active_loans())
    # The remaining loans in the log are returned.
    returned_loans = len(borrow_log) - open_loans
    # Collect the names of the most popular genre or genres.
    popular = []
    # Look at the popularity report.
    for row in most_popular_genre():
        # Keep the genre name.
        popular.append(row['genre'])
    # Build the summary as labels and values.
    summary = {}
    # Total number of books.
    summary['Total books'] = len(books)
    # Books that can be issued now.
    summary['Available books'] = available
    # Books currently issued.
    summary['Issued books'] = issued
    # Books that were already issued in the CSV, with no borrower details.
    summary['Source-issued books'] = source_issued
    # Members on record.
    summary['Registered members'] = len(members)
    # Members who hold at least one book.
    summary['Members holding books'] = len(members_with_books())
    # All loans recorded by this program.
    summary['Loans recorded'] = len(borrow_log)
    # Loans not yet returned.
    summary['Open loans'] = open_loans
    # Loans returned.
    summary['Returned loans'] = returned_loans
    # The most popular genre or genres, joined into one label.
    summary['Most popular issued genre'] = ', '.join(popular)
    # Give back the summary.
    return summary
# Print the summary one line per item.
for label, value in library_summary().items():
    # Print the label and its value.
    print(label + ':', value)

Total books: 63
Available books: 39
Issued books: 24
Source-issued books: 22
Registered members: 2
Members holding books: 2
Loans recorded: 4
Open loans: 2
Returned loans: 2
Most popular issued genre: Economics, Education, Fiction, History


### 6.14 Reconcile the final totals

In [74]:
# Get the final summary once.
summary = library_summary()
# Every book is either available or issued.
books_add_up = summary['Available books'] + summary['Issued books'] == summary['Total books']
# Print the first reconciliation.
print('Available + issued = total books:', books_add_up)
# Add the source-issued books and the open loans.
expected_issued = summary['Source-issued books'] + summary['Open loans']
# Issued books must equal that sum.
issued_add_up = summary['Issued books'] == expected_issued
# Print the second reconciliation.
print('Source-issued + open loans = issued books:', issued_add_up)
# Every recorded loan is either open or returned.
loans_add_up = summary['Loans recorded'] == summary['Open loans'] + summary['Returned loans']
# Print the third reconciliation.
print('Open + returned = loans recorded:', loans_add_up)

Available + issued = total books: True
Source-issued + open loans = issued books: True
Open + returned = loans recorded: True


## 7. Technical Notes

**Data structures.** Books and members are dictionaries keyed by whole-number IDs, so a lookup by ID is direct. The borrow log is a list of dictionaries with a loan ID, member ID, book ID and a returned flag. Pandas is used only to read and inspect the CSV and to print report tables; the library operations use plain Python.

**Validation rules.** IDs and ages must be positive whole numbers; text that looks like a number and Boolean values are refused. Names, contacts and book details must be non-empty text. Contact information is stored as text without assuming one format. Every rejected request returns a message and changes nothing.

**Borrowing rules.** Each Book ID is one copy. A book is issued only to a registered member and only when it is available. A return needs the same member and book in an open loan; this refuses the wrong member, a repeated return and the books issued in the source CSV. Completed loans stay in the log, so a returned book can be issued again without losing its history.

**Assumptions about the source.** The CSV marks 22 books as issued but gives no borrower details. They stay issued, count towards the popular-genre report and cannot be returned through the program until their borrower is known. Searches ignore letter case and outer spaces; genres are counted as written in the inventory. Popularity means the number of books issued right now, with all tied genres reported.

**Scope.** All data lives in memory for one session and the source CSV is never written back. Reloading Section 1 starts a fresh prototype state.

## 8. Conclusion and Limitations

The sample run covers adding books, registering members, borrowing several books, refusing invalid requests, returning, reissuing and all four reports. At the end of the run the library has **63 books: 39 available and 24 issued**, of which 22 were issued in the source CSV and 2 are on loan through the program. There are **2 registered members**, both holding a book, and **4 recorded loans: 2 open and 2 returned**. Every rejected request left the records unchanged, and the available, issued and loan totals reconcile. **Economics, Education, Fiction and History** share the highest number of issued books, at 3 each.

**Limitations**

1. Records live in memory for one session; nothing is saved back to a file.
2. Each Book ID is one copy, so the same title with several copies needs several IDs.
3. There are no due dates, fines or borrowing limits.
4. Member details cannot be edited or removed after registration.
5. The 22 books issued in the source CSV have no borrower, so they cannot be returned through the program until that information is added.
6. Contact information is stored as free text and is not checked for a valid format.

The optional interactive run in Appendix A adds records typed at the keyboard after this point, so its final summary differs from the figures above.

## Appendix A: Optional Interactive Run

This optional section takes keyboard input: a visitor checks whether a book is available, and a librarian adds a new book. It reuses the search and add functions from the sections above, so the same validation rules apply. It is placed after the Conclusion so that typed records do not change the figures of the tested sample run; the summary at the end of this appendix shows the counts after the entries. Each input cell waits for a value when the notebook runs.

### A.1 Check the availability of a book

In [75]:
# Define a report of availability for the books matching a title or author.
def check_book_availability(query):
    # Reuse the search function.
    matches = search_books(query)
    # Start with an empty result list.
    result = []
    # Look at every matching book.
    for book in matches:
        # Keep only the fields a visitor needs.
        record = {}
        # Store the book ID.
        record['book_id'] = book['book_id']
        # Store the title.
        record['title'] = book['title']
        # Store the author.
        record['author'] = book['author']
        # Store the availability.
        record['availability'] = book['availability']
        # Add the record to the results.
        result.append(record)
    # Give back the availability records.
    return result
# Ask for a full or partial title or author.
book_query = input('Enter a book title or author to check availability: ')
# Show the matching books and their availability.
show_table(check_book_availability(book_query))

Enter a book title or author to check availability: Harry Potter
 book_id        title      author availability
      48 Harry Potter J.K.Rowling    available


### A.2 Add a new book from keyboard input

In [76]:
# Define the interactive addition of a book.
def add_book_from_input():
    # Ask for the Book ID as text and remove outer spaces.
    id_text = input('Enter the new Book ID: ').strip()
    # The ID must contain digits only.
    if not id_text.isdigit():
        # Refuse anything that is not a whole number.
        return 'Rejected: Book ID must be a positive whole number.'
    # Convert the digits into a number.
    book_id = int(id_text)
    # Ask for the title.
    title = input('Enter the book title: ')
    # Ask for the author.
    author = input('Enter the author name: ')
    # Ask for the genre.
    genre = input('Enter the genre: ')
    # Reuse add_book so the same validation rules apply.
    return add_book(book_id, title, author, genre)
# Run the interactive addition and show the result.
print(add_book_from_input())

Enter the new Book ID: 62
Enter the book title: Diary of a Wimpy Kid
Enter the author name: Jeff Kinney
Enter the genre: Comedy
Added book 62: Diary of a Wimpy Kid.


### A.3 Verify the added book

In [77]:
# Ask which Book ID to look up and remove outer spaces.
id_text = input('Enter the Book ID to verify: ').strip()
# Check that the entry is a whole number.
if not id_text.isdigit():
    # Explain the problem with the entry.
    print('Book ID must be a positive whole number.')
# Check that the number is in the inventory.
elif int(id_text) not in books:
    # Explain that no such book exists.
    print('No book exists with this Book ID.')
# Otherwise the book exists.
else:
    # Show the record with its ID.
    print('Book ' + id_text + ':', books[int(id_text)])

Enter the Book ID to verify: 62
Book 62: {'title': 'Diary of a Wimpy Kid', 'author': 'Jeff Kinney', 'genre': 'Comedy', 'availability': 'available'}


### A.4 Library summary after the interactive entries

In [78]:
# Print the summary again so the effect of the typed entries is visible.
for label, value in library_summary().items():
    # Print the label and its value.
    print(label + ':', value)

Total books: 64
Available books: 40
Issued books: 24
Source-issued books: 22
Registered members: 2
Members holding books: 2
Loans recorded: 4
Open loans: 2
Returned loans: 2
Most popular issued genre: Economics, Education, Fiction, History


**Further Scope of Work**

In this assignment, checking the availability of a book and adding a book are two use cases performed to show case the use cases although it is not part of the assignment.

All the 5 modules are built as per the specification in the quessioniare with an additional run to check two use cases as mentioned above.

